In [1]:
import numpy             as np
import CoolProp.CoolProp as cpcp
from CoolProp.CoolProp   import PropsSI, set_reference_state



supported_fluids = cpcp.FluidsList()


def fluid_prop_calc_dict(fluid=None):
    """
    Calculate a comprehensive set of fluid properties (critical, triple point,
    limits, and other constants) provided by CoolProp.
    """
    if fluid is None:
        raise ValueError("fluid parameter is required")
    
    # Critical point properties
    T_crit = PropsSI('TCRIT', fluid)          # Critical temperature [K]
    P_crit = PropsSI('PCRIT', fluid)          # Critical pressure [Pa]
    rho_crit = PropsSI('RHOCRIT', fluid)      # Critical density [kg/m³]
    
    # Triple point (may fail for some mixtures)
    try:
        T_triple = PropsSI('TTRIPLE', fluid)  
        P_triple = PropsSI('PTRIPLE', fluid)  
    except Exception:
        T_triple = np.nan
        P_triple = np.nan
    
    # Limits
    T_min = PropsSI('TMIN', fluid)            
    T_max = PropsSI('TMAX', fluid)            
    P_max = PropsSI('PMAX', fluid)            
    
    # Fluid constants
    M = PropsSI('molemass', fluid)                   # Molar mass [kg/mol]
    R_specific = PropsSI('GAS_CONSTANT', fluid)          # Specific gas constant [J/kg·K]  (for ideal gas)
    acentric = PropsSI('ACENTRIC', fluid)     # Acentric factor [-]
    
    return {
        'fluid': fluid,
        'T_crit': T_crit,
        'P_crit': P_crit,
        'rho_crit': rho_crit,
        'T_triple': T_triple,
        'P_triple': P_triple,
        'T_min': T_min,
        'T_max': T_max,
        'P_max': P_max,
        'M': M,
        'R_specific': R_specific,
        'acentric_factor': acentric
    }


def state_prop_calc_dict(point=None, fluid=None, prop1=None, val1=None, prop2=None, val2=None, P_0=101325, T_0=288.15, ref='DEF'):
    """
    Calculate thermodynamic properties and specific flow exergy for a fluid state.
    
    Parameters:
        point (int, str, optional): Point identifier. If None → returns P, T, h, ... 
                                    If given → returns P_1, T_1, h_1, ... (or P_inlet, etc.)
        fluid (str): CoolProp fluid name (e.g. 'Water', 'R134a', 'Air')
        prop1, val1, prop2, val2: Input pair for state definition 
                                  (example: 'T', 300, 'P', 101325)
        P_0 (float): Reference pressure for exergy calculation [Pa], default = 101325 Pa (1 atm)
        T_0 (float): Reference temperature for exergy calculation [K], default = 288.15 K (15°C)
        ref (str): Reference state type for enthalpy (H) and entropy (S). 
                   Options in CoolProp:
                   - 'DEF'    : Default reference state of the fluid (recommended for most cases)
                   - 'IIR'    : h = 200 kJ/kg, s = 1 kJ/kg·K at 0°C saturated liquid
                   - 'ASHRAE' : h = 0, s = 0 at -40°C saturated liquid
                   - 'NBP'    : h = 0, s = 0 at Normal Boiling Point (1 atm saturated liquid)
    """
    
    if fluid is None or prop1 is None or val1 is None or prop2 is None or val2 is None:
        raise ValueError("fluid, prop1, val1, prop2, val2 parameters are required")
    
    # Set the reference state for enthalpy and entropy
    set_reference_state(fluid, ref)                     # Changes h and s reference for this fluid
    
    # Calculate thermodynamic properties at the given state
    P = PropsSI('P', prop1, val1, prop2, val2, fluid)  # Pressure [Pa]
    T = PropsSI('T', prop1, val1, prop2, val2, fluid)  # Temperature [K]
    h = PropsSI('H', prop1, val1, prop2, val2, fluid)  # Specific enthalpy [J/kg]
    s = PropsSI('S', prop1, val1, prop2, val2, fluid)  # Specific entropy [J/kg·K]
    u = PropsSI('U', prop1, val1, prop2, val2, fluid)  # Specific internal energy [J/kg]
    
    try:
        Q = PropsSI('Q', prop1, val1, prop2, val2, fluid)  # Vapor quality (dryness fraction) [-]
    except Exception:
        Q = np.nan                                         # Not defined (e.g. supercritical region)
    
    # Reference state properties (used for exergy)
    h_0 = PropsSI('H', 'T', T_0, 'P', P_0, fluid)          # Enthalpy at dead state [J/kg]
    s_0 = PropsSI('S', 'T', T_0, 'P', P_0, fluid)          # Entropy at dead state [J/kg·K]
    
    # Specific flow exergy (physical exergy)
    e = h - h_0 - T_0 * (s - s_0)                          # Specific flow exergy [J/kg]

    # Create dictionary keys with or without point suffix
    suffix = '' if point is None else f'_{point}'

    return {
        f'P{suffix}': P,
        f'T{suffix}': T,
        f'Q{suffix}': Q,
        f'u{suffix}': u,
        f'h{suffix}': h,
        f's{suffix}': s,
        f'e{suffix}': e
    }



state_prop_calc_dict(fluid='Water', prop1='T', val1=300, prop2='P', val2=101325, P_0=101325, T_0=288.15, ref='DEF')

fluid_prop_calc_dict('Water')

{'fluid': 'Water',
 'T_crit': 647.0959999999873,
 'P_crit': 22063999.999997754,
 'rho_crit': 321.9999990775634,
 'T_triple': 273.16,
 'P_triple': 611.6548008968684,
 'T_min': 273.16,
 'T_max': 2000.0,
 'P_max': 1000000000.0,
 'M': 0.018015268,
 'R_specific': 8.314371357587,
 'acentric_factor': 0.3442920843}

In [2]:
state_prop_calc_dict(fluid='R23', prop1='T', val1=25+273.15, prop2='P', val2=101325, P_0=101325, T_0=288.15, ref='DEF')

{'P': 101325.0,
 'T': 298.15,
 'Q': -1.0,
 'u': 361924.9288897883,
 'h': 397058.0643684717,
 's': 2065.5685573535507,
 'e': 124.18172718261212}